# Assignment 2 · Editable handwritten digits with TA-TiTok and MaskGen-L

В каждой паре сравниваем настоящую обучающую цифру MNIST из A1 и новый вариант её написания. Сначала ноутбук покажет **10 примеров**, по одному на класс. Посмотрите на последнюю миниатюру в каждой строке — именно такой вход 32×32 видит распознаватель.

**В Colab** выберите Runtime → Change runtime type → GPU и запустите все ячейки. По умолчанию запуск заканчивается на предпросмотре. Если цифры хорошо читаются, в разделе полной генерации включите `GENERATE_REMAINING` и запустите эту и последующие ячейки. Получится 12 вариантов каждого класса, 120 кандидатов. Для восстановления после отключения сессии включите `SAVE_TO_DRIVE` до первого запуска.

Мы взяли по 12 разных рукописных образцов каждого класса только из обучающей части A1. Вариант формы задаём небольшим поворотом, утончением или утолщением штриха. TA-TiTok кодирует полученный рисунок в 128 токенов. MaskGen-L заменяет 4–8 из них. TA-TiTok восстанавливает рисунок до и после замены, а программа добавляет разницу рядом со штрихом. Так сохраняются форма цифры и чистый фон. Перед обучением проверьте получившиеся цифры.

[Код и веса авторов](https://github.com/bytedance/1d-tokenizer/blob/942a96fbdd873780179d1b78d5462911528bf8c8/README_MaskGen.md) · [Статья](https://arxiv.org/abs/2501.07730) · [Источники и лицензия MNIST](https://keras.io/api/datasets/mnist/). Исходные PNG из A1 и их производные распространяются по CC BY-SA 3.0. Веса предназначены для исследовательского использования.

## 1. Подготовить среду Colab

Публичные веса занимают около 5,7 ГБ. Используем FP32 и обрабатываем по одному изображению.

In [ ]:
from pathlib import Path
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Runtime → Change runtime type, reconnect and run again.")

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
BASE_SEED = 42
CODE_REV = "942a96fbdd873780179d1b78d5462911528bf8c8"
CODE_URL = "https://github.com/bytedance/1d-tokenizer.git"
DEVICE = "cuda"
WORK = Path("/content")
SOURCE = WORK / f"1d-tokenizer-{CODE_REV[:10]}"

packages = [
    "open_clip_torch==2.30.0", "diffusers==0.32.2", "omegaconf==2.3.0",
    "huggingface_hub==0.34.4", "timm==1.0.15", "transformers==4.49.0",
    "accelerate==1.4.0", "peft==0.14.0", "einops>=0.7,<1", "safetensors>=0.4.5,<1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

def pinned_checkout(url, revision, destination, subfolder):
    destination.mkdir(parents=True, exist_ok=True)
    git = ["git", "-C", str(destination)]
    if not (destination / ".git").exists():
        subprocess.run([*git, "init", "-q"], check=True)
    head = subprocess.run([*git, "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if head.stdout.strip() != revision:
        subprocess.run([*git, "fetch", "-q", "--depth=1", "--filter=blob:none", url, revision], check=True)
        subprocess.run([*git, "sparse-checkout", "set", subfolder], check=True)
        subprocess.run([*git, "checkout", "-q", "--detach", revision], check=True)
    actual_revision = subprocess.check_output([*git, "rev-parse", "HEAD"], text=True).strip()
    if actual_revision != revision:
        raise RuntimeError(f"Expected {revision}, found {actual_revision}")
    return destination / subfolder

pinned_checkout(CODE_URL, CODE_REV, SOURCE, "modeling")
sys.path.insert(0, str(SOURCE))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = WORK / "drive/MyDrive/CV_Assignment2"
else:
    OUTPUT_ROOT = WORK / "assignment2_output"
print(torch.cuda.get_device_name(0), "| outputs:", OUTPUT_ROOT)

## 2. Выбрать исходные цифры из A1

Используем первые 12 строк обучающей части каждого класса в зафиксированном манифесте A1 (порядок уже перемешан). У каждого варианта свой оригинал. Сохраняем путь, индекс и хэш пикселей. Примеры для validation и test остаются в своих выборках.

In [ ]:
import csv
import hashlib
from pathlib import Path
from PIL import Image

A1_REV = "94ddf373e2b328bfa47f57718b33c3deb32b1a9e"
if "DATA_ROOT" not in globals():
    DATA_ROOT = pinned_checkout(
        "https://github.com/d0kuzu/Computer-Vision-project.git", A1_REV,
        WORK / f"a1-{A1_REV[:10]}", "assignment1/data/mnist_subset",
    )
with (DATA_ROOT / "manifest.csv").open(encoding="utf-8", newline="") as handle:
    a1_manifest = list(csv.DictReader(handle))

TRAIN_REFS = {}
for digit in range(10):
    refs = [r for r in a1_manifest if r["split"] == r["source_partition"] == "train"
            and r["class_name"] == r["label"] == str(digit)][:12]
    if len(refs) != 12:
        raise ValueError(f"Missing training references for {digit}")
    TRAIN_REFS[digit] = refs
if len({r["sha256"] for refs in TRAIN_REFS.values() for r in refs}) != 120:
    raise ValueError("Duplicate training references")

def load_reference(item):
    with Image.open(DATA_ROOT / item["source_relative_path"]) as original:
        gray = original.convert("L")
    if gray.size != (28, 28) or hashlib.sha256(gray.tobytes()).hexdigest() != item["source_sha256"]:
        raise ValueError(f"Changed A1 reference: {item['source_relative_path']}")
    return gray

## 3. Синтез с сохранением формы цифры

Реализация использует предобученные MaskGen-L VQ и TA-TiTok BL128 VQ. Параметры работы со штрихами и параметры предсказания токенов записываются вместе с результатами.

In [ ]:
"""Source-guided MaskGen variants for isolated digits.

The reference controls the glyph geometry. Morphology or rotation supplies the
requested style. MaskGen refills selected TA-TiTok tokens, and the change it
makes to the decoded ink is applied locally to the styled reference.
"""
import numpy as np
from PIL import Image, ImageFilter, ImageOps
import torch


METHOD = "reference_masked_tokens_v1"
METHOD_CONFIG = {
    "method": METHOD, "canvas_px": 256, "ink_box_px": 186, "ink_threshold": 40,
    "thin_radii": [1, 2, 3, 2], "thin_mix": [0.80, 0.85, 0.70, 1.0],
    "bleed_radii": [2, 3, 2, 4], "bleed_blur_sigmas": [0.65, 0.90, 0.45, 1.15],
    "rotation_ccw_degrees": [-8, 8, -14, 14], "masked_counts": [4, 6, 8, 5],
    "sampler_top_k": 8, "sampler_temperature": 0.8, "aesthetic_score": 6.5,
    "residual_clip_gray": 64, "halo_radius_px": 5, "halo_blur_sigma_px": 1.5,
}


def prepare_reference(source):
    """Center one dark glyph on a 256-pixel white field, preserving aspect ratio."""
    raw = ImageOps.exif_transpose(source).convert("RGBA")
    background = Image.new("RGBA", raw.size, "white")
    background.alpha_composite(raw)
    gray = background.convert("L")
    pixels = np.asarray(gray)
    border = np.concatenate((pixels[0], pixels[-1], pixels[:, 0], pixels[:, -1]))
    if np.median(border) < 128:
        gray = ImageOps.invert(gray)
    ink = ImageOps.invert(gray)
    box = ink.point(lambda value: 255 if value > METHOD_CONFIG["ink_threshold"] else 0).getbbox()
    if not box:
        raise ValueError("The reference contains no visible digit.")
    cropped = ink.crop(box)
    scaled = ImageOps.contain(cropped, (METHOD_CONFIG["ink_box_px"],) * 2, method=Image.Resampling.LANCZOS)
    side = METHOD_CONFIG["canvas_px"]
    field = Image.new("L", (side, side), 0)
    field.paste(scaled, ((side - scaled.width) // 2, (side - scaled.height) // 2))
    return ImageOps.invert(field).convert("RGB")


def style_reference(reference, variant):
    """Apply a known, recorded change in stroke thickness or glyph angle."""
    if not (1 <= variant <= 12):
        raise ValueError("Expected a variant number in 1..12.")
    ink = ImageOps.invert(reference.convert("L"))
    index = (variant - 1) % 4
    if variant <= 4:
        radius = METHOD_CONFIG["thin_radii"][index]
        amount = METHOD_CONFIG["thin_mix"][index]
        eroded = ink.filter(ImageFilter.MinFilter(2 * radius + 1))
        ink = Image.blend(ink, eroded, amount)
        control = dict(style="thin-brush", erosion_radius_px=radius, erosion_mix=amount)
    elif variant <= 8:
        radius = METHOD_CONFIG["bleed_radii"][index]
        blur = METHOD_CONFIG["bleed_blur_sigmas"][index]
        ink = ink.filter(ImageFilter.MaxFilter(2 * radius + 1)).filter(ImageFilter.GaussianBlur(blur))
        control = dict(style="marker-bleed", dilation_radius_px=radius, blur_sigma_px=blur)
    else:
        angle = METHOD_CONFIG["rotation_ccw_degrees"][index]
        ink = ink.rotate(angle, resample=Image.Resampling.BICUBIC, fillcolor=0)
        control = dict(style="rotation", rotation_ccw_degrees=angle)
    return ImageOps.invert(ink).convert("RGB"), control


@torch.inference_mode()
def generate_variant(source, item, generator, tokenizer, clip, text_tokenizer, device, *, stages=None):
    """Return an RGB 256×256 image and JSON-compatible diagnostics.

    Set `stages={}` to also inspect the styled reference and both raw decoder
    outputs. A model, shape or numerical failure raises and produces no image.
    """
    from modeling.maskgen import open_clip_text_encoding

    variant, seed = int(item["variant"]), int(item["seed"])
    reference = prepare_reference(source)
    styled, control = style_reference(reference, variant)
    image = np.asarray(styled, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).to(device)
    _, quantized = tokenizer.encode(tensor)
    codes = quantized["min_encoding_indices"].reshape(1, -1).long()
    if codes.shape != (1, 128):
        raise RuntimeError(f"Unexpected TA-TiTok code shape {tuple(codes.shape)}")

    prompt = str(item["prompt"])
    context, pooled = open_clip_text_encoding(text_tokenizer, clip, [prompt])
    words = text_tokenizer([prompt]).to(device)
    guidance = clip.ln_final(clip.transformer(clip.token_embedding(words) + clip.positional_embedding,
                                             attn_mask=clip.attn_mask))
    # Seed on CPU for the same mask positions and multinomial draws on CPU/CUDA.
    rng = torch.Generator(device="cpu").manual_seed(seed)
    count = METHOD_CONFIG["masked_counts"][(variant - 1) % 4]
    positions = torch.randperm(128, generator=rng)[:count].sort().values.to(device)
    masked = codes.clone()
    masked[:, positions] = generator.mask_token_id
    score = torch.full((1,), METHOD_CONFIG["aesthetic_score"], device=device)
    logits, _ = generator(masked, context, pooled, aesthetic_score=score)
    logits = logits[0, positions].float()
    if not torch.isfinite(logits).all().item():
        raise RuntimeError("Non-finite MaskGen logits.")
    # Preserve every unmasked code. Among the eight highest-scoring available
    # alternatives, sample one new code for each selected position.
    logits.scatter_(1, codes[0, positions].unsqueeze(1), float("-inf"))
    values, indices = logits.topk(METHOD_CONFIG["sampler_top_k"], dim=-1)
    probabilities = (values / METHOD_CONFIG["sampler_temperature"]).softmax(-1).cpu()
    drawn = torch.multinomial(probabilities, 1, generator=rng).to(device)
    edited = codes.clone()
    edited[0, positions] = indices.gather(1, drawn).squeeze(1)

    # Reconstruct both code sequences under the SAME decoder text. Their
    # difference isolates the effect of resampling the selected MaskGen codes.
    pair = tokenizer.decode_tokens(torch.cat([codes, edited]), guidance.expand(2, -1, -1))
    if pair.shape != (2, 3, 256, 256) or not torch.isfinite(pair).all().item():
        raise RuntimeError("TA-TiTok returned invalid decoded pixels.")
    rgb = (pair.clamp(0, 1).mul(255).round().byte().permute(0, 2, 3, 1).cpu().numpy())
    base_recon, edited_recon = (Image.fromarray(value) for value in rgb)
    styled_ink = ImageOps.invert(styled.convert("L"))
    # At full intensity the local learned perturbation may change up to 64
    # grayscale levels. Beyond a 5 px dilated/softened stroke halo it vanishes.
    halo = styled_ink.filter(ImageFilter.MaxFilter(METHOD_CONFIG["halo_radius_px"] * 2 + 1))
    envelope = np.asarray(halo.filter(ImageFilter.GaussianBlur(METHOD_CONFIG["halo_blur_sigma_px"])),
                          dtype=np.float32) / 255.0
    anchor = np.asarray(styled_ink, dtype=np.float32)
    ink_difference = (np.asarray(base_recon.convert("L"), dtype=np.float32) -
                      np.asarray(edited_recon.convert("L"), dtype=np.float32))
    limit = METHOD_CONFIG["residual_clip_gray"]
    residual = envelope * np.clip(ink_difference, -limit, limit)
    final_ink = np.clip(np.rint(anchor + residual), 0, 255)
    result = Image.fromarray((255 - final_ink).astype(np.uint8)).convert("RGB")
    delta = abs(final_ink - anchor)
    support = envelope > 0.05
    # A numerically inert model edit is surfaced, never silently substituted.
    if not np.any(delta >= 1):
        raise RuntimeError("MaskGen token edit produced no visible pixel change.")
    info = dict(method=METHOD, control=control, seed=seed, mask_indices=positions.tolist(),
                original_codes=codes[0, positions].tolist(), replacement_codes=edited[0, positions].tolist(),
                changed_code_count=int((codes != edited).sum().item()),
                sampler_top_k=METHOD_CONFIG["sampler_top_k"], sampler_temperature=METHOD_CONFIG["sampler_temperature"],
                aesthetic_score=METHOD_CONFIG["aesthetic_score"], residual_clip_gray=limit,
                halo_radius_px=METHOD_CONFIG["halo_radius_px"], halo_blur_sigma_px=METHOD_CONFIG["halo_blur_sigma_px"],
                mean_absolute_change_on_stroke=float(delta[support].mean()),
                changed_pixels=int(np.count_nonzero(delta)), max_pixel_change=int(delta.max()))
    if stages is not None:
        stages.update(reference=reference, styled=styled, reconstruction=base_recon,
                      edited_decode=edited_recon)
    return result, info


## 4. Составить план эксперимента

В пилоте представлены тонкие штрихи, растекание и поворот. Смена метода или его параметров создаёт новый каталог запуска.

In [ ]:
import hashlib
import json
import platform
import os
import tempfile
from contextlib import contextmanager
from importlib.metadata import version
from PIL import Image

@contextmanager
def staged_path(destination):
    # Publish a complete file only after the write succeeds.
    with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as handle:
        staging = Path(handle.name)
    try:
        yield staging
        os.replace(staging, destination)
    finally:
        staging.unlink(missing_ok=True)

WEIGHTS = {
    "maskgen": ["turkeyju/generator_maskgen_vq_l", "d3fbbe36c3223ab14d6b0019c04eba971a5f8bd4"],
    "tatitok": ["turkeyju/tokenizer_tatitok_bl128_vq", "7650d466a955c754265d20e57513518a4e049f00"],
    "clip": ["timm/vit_large_patch14_clip_336.openai", "81e38efc4637de5023b10e75a7f9bd1c6fa6b010"],
}
DIGIT_NAMES = "zero one two three four five six seven eight nine".split()
STYLES = ["thin-brush"] * 4 + ["marker-bleed"] * 4 + ["rotation"] * 4
METHOD_SOURCE_SHA256 = "da30fddd65cb0914a98b5c47f6ab2f0608bb3772c7f08e0a4300cb31a57ffaca"
PLAN = []
for digit, name in enumerate(DIGIT_NAMES):
    for variant, style in enumerate(STYLES, start=1):
        ref = TRAIN_REFS[digit][variant - 1]
        sample_id = f"d{digit}-v{variant:02d}"
        prompt = f"A single handwritten digit {digit}, isolated black strokes on a plain white field."
        PLAN.append(dict(
            sample_id=sample_id, requested_digit=digit, variant=variant, style=style,
            prompt=prompt, seed=BASE_SEED + 100 * digit + variant,
            path=f"candidates/{sample_id}.png", source="synthetic",
            source_relative_path=ref["relative_path"], source_sha256=ref["sha256"],
            source_split=ref["split"], source_partition=ref["source_partition"],
            source_index=int(ref["source_index"]),
        ))
PILOT_IDS = [f"d{digit}-v{(1, 6, 11)[digit % 3]:02d}" for digit in range(10)]
PILOT = [item for item in PLAN if item["sample_id"] in PILOT_IDS]

runtime = {
    "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if str(DEVICE) == "cuda" else None,
    "cuda": torch.version.cuda, "python": platform.python_version(),
    "packages": {p: version(p) for p in ("torch", "torchvision", "open_clip_torch", "diffusers",
                                         "huggingface_hub", "omegaconf", "timm", "safetensors", "pillow")},
}
recipe = dict(method="reference_masked_tokens_v1", implementation_sha256=METHOD_SOURCE_SHA256,
              edit=METHOD_CONFIG, source_revision=CODE_REV, source_url=CODE_URL, weights=WEIGHTS,
              a1_revision=A1_REV, reference_subset="A1 train", base_seed=BASE_SEED,
              precision="float32", batch_size=1, runtime=runtime, samples=PLAN)
signature = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR = OUTPUT_ROOT / f"reference_maskgen_{signature}"
for folder in ("candidates", "diagnostics"):
    (RUN_DIR / folder).mkdir(exist_ok=True, parents=True)
run_meta = RUN_DIR / "run.json"
if not run_meta.exists():
    with staged_path(run_meta) as tmp:
        tmp.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
if json.loads(run_meta.read_text(encoding="utf-8")) != recipe:
    raise ValueError("Run directory belongs to a different recipe")
print(f"{len(PILOT)} pilot images · {len(PLAN)} planned · {RUN_DIR}")

## 5. Загрузить веса

У TA-TiTok используются и энкодер, и декодер. OpenCLIP обрабатывает текст, который сопровождает исходное изображение.

In [ ]:
import gc
import open_clip
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model
from modeling.maskgen import MaskGen_VQ
from modeling.tatitok import TATiTok

def weight_file(component, filename):
    repo, revision = WEIGHTS[component]
    return hf_hub_download(repo_id=repo, revision=revision, filename=filename)

def load_component(component, model_type):
    config = json.loads(Path(weight_file(component, "config.json")).read_text(encoding="utf-8"))
    model = model_type(config)
    load_model(model, weight_file(component, "model.safetensors"), strict=True)
    return model.requires_grad_(False).eval().to(DEVICE)

def complete_png(path):
    try:
        with Image.open(path) as im:
            im.load()
            return im.format == "PNG" and im.mode == "RGB" and im.size == (256, 256)
    except (OSError, ValueError):
        return False

def complete_candidate(item):
    path = RUN_DIR / item["path"]
    details = RUN_DIR / "diagnostics" / (item["sample_id"] + ".json")
    if not complete_png(path) or not details.is_file():
        return False
    try:
        record = json.loads(details.read_text(encoding="utf-8"))
        return record["sha256"] == hashlib.sha256(path.read_bytes()).hexdigest()
    except (OSError, ValueError, KeyError, TypeError):
        return False

if any(not complete_candidate(item) for item in PLAN):
    clip = open_clip.create_model(
        "ViT-L-14-336", pretrained=weight_file("clip", "open_clip_model.safetensors"),
        force_quick_gelu=True, device="cpu",
    )
    del clip.visual
    clip = clip.requires_grad_(False).eval().to(DEVICE)
    text_tokenizer = open_clip.get_tokenizer("ViT-L-14-336")
    generator = load_component("maskgen", MaskGen_VQ)
    tokenizer = load_component("tatitok", TATiTok)
    gc.collect()
    assert generator.image_seq_len == tokenizer.num_latent_tokens == 128
    assert tokenizer.quantize_mode == "vq" and clip.transformer.batch_first
    print("MaskGen-L VQ, TA-TiTok and CLIP loaded.")
else:
    print("All 120 PNGs already exist for this recipe. Continuing to the gallery and export.")

In [ ]:
def ensure_candidate(item):
    path = RUN_DIR / item["path"]
    if complete_candidate(item):
        return path
    image, diagnostics = generate_variant(
        load_reference(item), item, generator, tokenizer, clip, text_tokenizer, DEVICE
    )
    if image.mode != "RGB" or image.size != (256, 256):
        raise ValueError("Expected an RGB image at 256 × 256.")
    with staged_path(path) as tmp:
        image.save(tmp, format="PNG")
    info = {**diagnostics, "sample_id": item["sample_id"],
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    with staged_path(RUN_DIR / "diagnostics" / (item["sample_id"] + ".json")) as tmp:
        tmp.write_text(json.dumps(info, indent=2), encoding="utf-8")
    return path

## 6. Посмотреть первые 10 результатов

В каждой карточке слева направо: оригинал из A1, сгенерированный вариант и его вид после уменьшения для CNN. Найдите цифру именно на последней миниатюре. Если она неоднозначна, это повод отклонить конкретный результат.

In [ ]:
from PIL import ImageDraw, ImageOps

def a1_preview(image):
    ink = ImageOps.invert(image.convert("L"))
    ink = ImageOps.contain(ink, (24, 24), method=Image.Resampling.LANCZOS)
    canvas = Image.new("L", (32, 32), 0)
    canvas.paste(ink, ((32 - ink.width) // 2, (32 - ink.height) // 2))
    return canvas

def contact_sheet(rows, columns):
    w, h = 220, 106
    sheet = Image.new("RGB", (columns * w, ((len(rows) + columns - 1) // columns) * h), "#eef1f5")
    draw = ImageDraw.Draw(sheet)
    for i, row in enumerate(rows):
        x, y = (i % columns) * w, (i // columns) * h
        ref = load_reference(row)
        with Image.open(RUN_DIR / row["path"]) as image:
            sheet.paste(ref.resize((64, 64), Image.Resampling.LANCZOS), (x + 4, y + 5))
            sheet.paste(image.resize((64, 64), Image.Resampling.LANCZOS), (x + 73, y + 5))
            sheet.paste(a1_preview(image).resize((64, 64), Image.Resampling.NEAREST), (x + 142, y + 5))
        draw.text((x + 4, y + 72), row["sample_id"] + "   " + row["style"], fill="#18232f")
        draw.text((x + 4, y + 87), "source          variant          CNN", fill="#455566")
    return sheet

In [ ]:
from IPython.display import display
from tqdm.auto import tqdm

for item in tqdm(PILOT, desc="Pilot", unit="digit"):
    ensure_candidate(item)
preview = contact_sheet(PILOT, columns=2)
with staged_path(RUN_DIR / "preview_10.png") as tmp:
    preview.save(tmp, format="PNG")
display(preview)

## 7. После просмотра запустить оставшиеся варианты

Чтобы дождаться полной серии, включите галочку **GENERATE_REMAINING** ниже и запустите эту и последующие ячейки. Десять уже сохранённых результатов будут использованы повторно.

In [ ]:
GENERATE_REMAINING = False  # @param {type:"boolean"}
if GENERATE_REMAINING:
    for item in tqdm(PLAN, desc="Complete batch", unit="image"):
        ensure_candidate(item)
else:
    print("Предпросмотр готов. Для всех 120 изображений включите GENERATE_REMAINING.")

## 8. Проверить метки и скачать архив

В ZIP входят 120 кандидатов, листы для просмотра, настройки и таблицы. Для каждой строки `review.csv` доступны статусы `pending`, `accept` и `reject`. Для принятого изображения укажите действительно видимую цифру в `reviewed_digit`. Проверяйте обрезанные штрихи и двусмысленные знаки в маленьком входе CNN. Контактные листы и метки можно просмотреть в демо A2 после распаковки.

In [ ]:
import csv
from zipfile import ZipFile, ZIP_DEFLATED

from PIL import Image, ImageDraw, ImageOps

def write_csv(destination, fields, rows):
    with staged_path(destination) as staging:
        with staging.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)


def export_pilot():
    candidates = sorted(PLAN, key=lambda r: (r["requested_digit"], r["variant"]))
    expected_pairs = [(d, v) for d in range(10) for v in range(1, 13)]
    if [(r["requested_digit"], r["variant"]) for r in candidates] != expected_pairs:
        raise ValueError("Expected 12 variants per digit (0–9). Check PLAN.")
    if len({r["sample_id"] for r in candidates}) != 120:
        raise ValueError("Expected 120 distinct sample IDs.")

    manifest_fields = ["sample_id", "requested_digit", "variant", "style", "prompt",
                       "seed", "path", "source", "source_relative_path", "source_sha256",
                       "source_index", "source_split", "source_partition", "sha256"]
    manifest_rows = []
    for row in candidates:
        expected_path = "candidates/d{}-v{:02d}.png".format(row["requested_digit"], row["variant"])
        if row["path"] != expected_path or row["source"] != "synthetic":
            raise ValueError(f"Unexpected path or source for {row['sample_id']}")
        path = RUN_DIR / expected_path
        with Image.open(path) as image:
            if (image.format, image.mode, image.size) != ("PNG", "RGB", (256, 256)):
                raise ValueError(f"Expected an RGB 256×256 PNG: {expected_path}")
            image.load()
        manifest_rows.append({**row, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})

    identity_fields = ["sample_id", "requested_digit", "path", "sha256"]
    review_fields = identity_fields + ["status", "reviewed_digit", "notes"]
    review_path = RUN_DIR / "review.csv"
    if not review_path.exists():
        write_csv(review_path, review_fields,
                  ({**row, "status": "pending", "reviewed_digit": "", "notes": ""}
                   for row in manifest_rows))

    def read_review():
        with review_path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            if not set(review_fields).issubset(reader.fieldnames or []):
                raise ValueError("review.csv is missing columns. See review_fields.")
            rows = list(reader)
        def identities(records):
            return sorted(tuple(str(row[key]) for key in identity_fields) for row in records)
        if identities(rows) != identities(manifest_rows):
            raise ValueError("review.csv refers to a different set of PNGs. Check IDs and hashes.")
        for row in rows:
            if row["status"] not in {"pending", "accept", "reject"}:
                raise ValueError(f"Invalid review status for {row['sample_id']}.")
            if row["reviewed_digit"] not in {"", *map(str, range(10))}:
                raise ValueError(f"Invalid reviewed_digit for {row['sample_id']}.")
            if row["status"] == "accept" and row["reviewed_digit"] == "":
                raise ValueError(f"Confirm a reviewed_digit for {row['sample_id']}.")
        return rows

    reviewed = read_review()
    write_csv(RUN_DIR / "manifest.csv", manifest_fields, manifest_rows)

    gallery_dir = RUN_DIR / "galleries"
    gallery_dir.mkdir(exist_ok=True)
    sheet_paths = []
    for digit in range(10):
        relative = f"galleries/digit-{digit}.png"
        rows = [row for row in manifest_rows if row["requested_digit"] == digit]
        with staged_path(RUN_DIR / relative) as tmp:
            contact_sheet(rows, columns=3).save(tmp, format="PNG")
        sheet_paths.append(relative)

    # One row per class, one column per variant, in the order used by the manifest.
    tile = 56
    overview = Image.new("RGB", (32 + 12 * tile, 48 + 10 * tile), "white")
    labels = ImageDraw.Draw(overview)
    for group, title in enumerate(("Thin stroke", "Thick stroke", "Rotation")):
        labels.text((36 + 4 * group * tile, 4), title, fill="black")
    for variant in range(1, 13):
        labels.text((52 + (variant - 1) * tile, 25), f"{variant:02d}", fill="#344457")
    for row in manifest_rows:
        digit, column = row["requested_digit"], row["variant"] - 1
        x, y = 32 + column * tile, 48 + digit * tile
        if column == 0:
            labels.text((10, y + 18), str(digit), fill="black")
        with Image.open(RUN_DIR / row["path"]) as image:
            overview.paste(image.resize((50, 50), Image.Resampling.LANCZOS), (x + 3, y + 3))
    with staged_path(RUN_DIR / "all_120.png") as tmp:
        overview.save(tmp, format="PNG")

    members = [r["path"] for r in manifest_rows] + sheet_paths + ["all_120.png"]
    members += [f"diagnostics/{r['sample_id']}.json" for r in manifest_rows]
    members += ["manifest.csv", "review.csv", "run.json", "preview_10.png", "SOURCE.txt"]
    (RUN_DIR / "SOURCE.txt").write_text(
        "Synthetic derivatives of MNIST training examples by Yann LeCun and Corinna Cortes, "
        "derived from NIST. Dataset license: CC BY-SA 3.0, "
        "https://creativecommons.org/licenses/by-sa/3.0/. "
        "Originals: https://keras.io/api/datasets/mnist/. "
        "Changes: stroke transforms, TA-TiTok tokenization, MaskGen-L partial resampling, "
        "decoding and source-constrained rendering. See run.json and per-image diagnostics.\n",
        encoding="utf-8",
    )
    archive_path = RUN_DIR / "ta_titok_reference_120.zip"
    with staged_path(archive_path) as tmp:
        with ZipFile(tmp, "w", compression=ZIP_DEFLATED) as bundle:
            for relative in members:
                bundle.write(RUN_DIR / relative, arcname=relative)
    print(f"{archive_path.name} · {len(manifest_rows)} кандидатов")
    return archive_path

In [ ]:
if all(complete_candidate(item) for item in PLAN):
    archive_path = export_pilot()
else:
    archive_path = None
    print("Полный архив появится после генерации всех 120 кандидатов.")

In [ ]:
if archive_path is not None:
    from google.colab import files
    files.download(str(archive_path))